# 이온 밀 식각 장비 예지보전 EDA — 20대 전체 보기 + 대표 장비 6대

| 항목 | 내용 |
|---|---|
| 목적 | 장비마다 패턴이 다른지, 고장 많은 장비와 적은 장비는 평소부터 다른지를 장비 단위로 자세히 본다 |
| 데이터 | `data/processed/step_s1/` (원본에서 셔터 1 행만 먼저 거른 뒤 레시피 스텝 구간으로 요약). 1장은 **20대 전체**, 2장부터는 **대표 6대 전체 구간**. 셔터 1 시간 비율만 `step/`(모든 셔터)으로 계산 |
| 과제 성격 | 모델 학습 없음. 고장 기록은 그림에 표시하고 고장 전·평소를 나누는 데만 쓴다 |
| 기준 노트북 | `eda/eda_step.ipynb` (평소·z값·고장 전 정의를 그대로 따른다) |

**대표 6대** — 고장 많은 3대 + 적은 3대. M01 3대 · M02 3대로 맞춰 "고장이 적어서"와 "M02라서"가 섞이지 않게 했다.

| 역할 | 장비 | 고장 기록 | 고른 이유 |
|---|---|---|---|
| 고장 최다 · 반대 패턴 | `03_M02` | 356 | 고장이 가장 많고, 압력 저하 전 냉각 압력이 **올라가는** 소수 묶음 |
| 고장 많은 M01 | `09_M01` | 150 | 고장 2위 |
| 고장 많은 M01 · 다수 패턴 | `08_M01` | 117 | 고장 3위, 압력 저하 전 냉각 압력이 **내려가는** 다수 묶음 |
| 고장 적은 M01 | `04_M01` | 9 | M01 중 고장이 가장 적음 |
| 고장 적은 M02 | `07_M02` | 6 | M02 중 고장이 적음 |
| 고장 최소 M02 | `10_M02` | 2 | 20대 중 고장이 가장 적음 |

고장 적은 3대는 사건이 몇 개뿐이라 고장 전 궤적(6장)은 사례 수준이다. 주로 **평소 상태 비교용**이다.

**그림 그릴 때 지킨 것**
- **무작위 샘플링 안 함.** 시계열 중간이 빠지면 고장 전 변화가 끊겨 보인다. 6대는 전체 구간을 그대로 쓴다.
- **셔터 1 구간만** 그린다(식각 중, 근거는 `eda_step` 2-3).
- **20대 전체 그림은 하루 평균·분위수로 요약**해 한 장에 그린다(1-2절).
- **긴 기간 그림은 6시간 평균**(원본 행 수 가중)으로 묶는다. 장비당 수천 점이라 가볍다.
- **분포 그림은 미리 계산한 분위수**(5·25·50·75·95%)로 상자를 그린다. 수십만 점을 그림에 넣지 않는다.
- **고장 전후 확대 그림만** 줄이지 않은 원래 해상도(스텝 구간 하나 = 점 하나)로 그린다.
- 그림은 파일로 저장하지 않고 노트북에만 표시한다.

## 0. 분석 목적

`eda_step`은 20대를 한꺼번에 보고 "어떤 센서가 고장과 관련 있는가"를 확인했다. 그런데 모델은 처음 보는 장비에서 AUC 0.5 근처에 머물렀다. 이 노트북은 **장비 단위로 들여다보며** 그 이유를 찾는다.

| 질문 | 보는 곳 | 이 답으로 정할 것 |
|---|---|---|
| Q1. 장비마다 고장 전 냉각 압력의 방향이 같은가? | 1장, 6장 | 모든 장비에 하나의 모델을 쓸지, 장비별(또는 M01/M02별)로 나눌지 |
| Q2. 냉각 압력 차이는 고장 직전에 생기는 변화인가, 원래부터 있던 수준 차이인가? | 3장, 6장, 7장 | 입력을 "현재 수준"으로 할지 "최근 대비 변화"로 할지 |
| Q3. 고장 많은 장비와 적은 장비는 평소 상태부터 다른가? | 4장 | 경보 기준을 모든 장비에 같게 둘지, 장비별로 둘지 |
| Q4. 장비끼리 공정(레시피)이 겹치는가? | 2장 | 처음 보는 장비로 일반화를 기대할 수 있는지 |
| Q5. 고장이 반복 기록되는 정도는 장비마다 다른가? | 5장 | 사건 정의(하루 묶음)와 직후 경보 규칙을 모든 장비에 쓸 수 있는지 |

답은 8장 해석에 정리한다.

In [1]:
# ── 1) 라이브러리 ──────────────────────────────────────────────
from pathlib import Path
import json
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

# ── 2) 경로: 노트북을 프로젝트 루트나 eda/ 어디서 열어도 data/processed를 찾는다 ──
cwd = Path.cwd().resolve()
ROOT = next(path for path in (cwd, cwd.parent) if (path / 'data/processed/step').exists())
PROC = ROOT / 'data/processed'

def show(fig, height=520):
    """그림은 파일로 저장하지 않고 노트북에만 표시한다."""
    fig.update_layout(template='plotly_white', height=height, font=dict(family='Malgun Gothic, sans-serif'),
                      margin=dict(l=30, r=30, t=75, b=70))
    display({'application/vnd.plotly.v1+json': json.loads(fig.to_json())}, raw=True)

# ── 3) 상수 ──────────────────────────────────────────────────────
# 이 노트북에서 보는 센서 6개 (냉각 계통 2개 + 공정 센서 4개)
KEY = ['FLOWCOOLPRESSURE', 'FLOWCOOLFLOWRATE', 'IONGAUGEPRESSURE', 'ETCHSUPPRESSORCURRENT', 'ETCHBEAMCURRENT', 'ETCHGASCHANNEL1READBACK']
NAMES = {'FLOWCOOLPRESSURE': '냉각 압력', 'FLOWCOOLFLOWRATE': '냉각 유량', 'IONGAUGEPRESSURE': '챔버 압력',
         'ETCHSUPPRESSORCURRENT': '억제 전류', 'ETCHBEAMCURRENT': '빔 전류', 'ETCHGASCHANNEL1READBACK': '아르곤 유량(소스)'}
# 고장 3종: 원래 이름 -> 짧은 이름
FAULTS = ['FlowCool Pressure Dropped Below Limit', 'Flowcool Pressure Too High Check Flowcool Pump', 'Flowcool leak']
FSHORT = dict(zip(FAULTS, ['압력 저하', '압력 과다', '누설']))
ORDER = [FSHORT[fault] for fault in FAULTS]
TTF = {fault: 'TTF_' + fault for fault in FAULTS}      # 고장까지 남은 시간(초) 열 이름
FEAT = [sensor + '_mean' for sensor in KEY]            # 스텝 구간 안 4초 값의 평균 열
HOUR, DAY = 3600, 86400                                # 시간 단위(초)
# 대표 6대(고장 많은 3대 + 적은 3대, M01 3 · M02 3)와 역할, 그림 색 (많은 쪽 = 붉은 계열, 적은 쪽 = 푸른 계열)
SEL = ['03_M02', '09_M01', '08_M01', '04_M01', '07_M02', '10_M02']
ROLE = dict(zip(SEL, ['고장 최다 · 반대 패턴', '고장 많은 M01', '고장 많은 M01 · 다수 패턴',
                      '고장 적은 M01', '고장 적은 M02', '고장 최소 M02']))
TCOLOR = dict(zip(SEL, ['#b42318', '#eb6834', '#f2a541', '#2a78d6', '#4c9a52', '#7a7f87']))
FCOLOR = dict(zip(ORDER, ['#2a78d6', '#eb6834', '#7a7f87']))
FSYMBOL = dict(zip(ORDER, ['triangle-down', 'triangle-up', 'x']))

def disp(tool):
    """장비 이름을 08_M01 형식으로 통일한다. (step 파일: M01_08, 고장 파일: 08M01)"""
    return f'{tool[4:]}_{tool[:3]}' if tool[3] == '_' else f'{tool[:2]}_{tool[2:]}'

# ── 4) 스텝 구간 데이터: 20대 전부, 필요한 열만 읽는다 (샘플링 없음) ──
# 분석 데이터 = step_s1/: 원본 4초 행에서 셔터 1(식각 중)만 먼저 거른 뒤 스텝 구간으로 요약 (eda_step 2-3)
load_cols = ['Tool', 'time_start', 'time_end', 'n_rows', 'recipe', 'recipe_step', 'shutter'] + list(TTF.values()) + FEAT
df = pd.concat([pd.read_parquet(path, columns=load_cols) for path in sorted((PROC / 'step_s1').glob('*.parquet'))],
               ignore_index=True)
df['Tool'] = df.Tool.map(disp)
df = df.sort_values(['Tool', 'time_start']).reset_index(drop=True)   # 장비별 시간순 정렬 (searchsorted에 필요)
# 셔터 전체 데이터(step/): 셔터 1 시간 비율·기록 기간 계산에만 쓴다
full_df = pd.concat([pd.read_parquet(path, columns=['Tool', 'time_start', 'time_end', 'n_rows', 'shutter'])
                     for path in sorted((PROC / 'step').glob('*.parquet'))], ignore_index=True)
full_df['Tool'] = full_df.Tool.map(disp)
tool_start_by = full_df.groupby('Tool').time_start.min()             # 장비 기록 시작 시각 (모든 셔터 기준)

# ── 5) 고장 기록 ─────────────────────────────────────────────────
faults = pd.read_csv(PROC / 'faults.csv')
faults['Tool'] = faults.Tool.map(disp)
faults['fault_type'] = faults.fault_name.str.replace(' [NoWaferID]', '', regex=False)   # 같은 고장의 표기 차이 통일
faults['종류'] = faults.fault_type.map(FSHORT)
# 같은 장비·같은 시각·같은 종류로 두 번 찍힌 기록은 하나로 본다
events = faults.drop_duplicates(['Tool', 'time', 'fault_type']).sort_values(['Tool', 'time']).reset_index(drop=True)
# 하루 묶음 사건: 같은 장비에서 앞 기록과 1일 넘게 떨어지면 새 사건(고장 종류 무관)
gap_from_prev = events.groupby('Tool').time.diff()
events['new'] = gap_from_prev.isna() | (gap_from_prev > DAY)
events['event_id'] = events.groupby('Tool').new.cumsum()
# 불러온 데이터가 예상과 다르면 여기서 멈춘다
assert len(faults) == 1236 and df.Tool.nunique() == 20 and (df.shutter == 1).all()

### 대표 6대가 전체에서 차지하는 비율

셔터 1 구간, 고장 기록(중복 제외), 하루 묶음 사건 세 가지로 대표 6대와 나머지 14대의 비율을 본다.

In [2]:
# ── 장비별 개수: 스텝 구간 수, 고장 기록 수(중복 제외), 하루 묶음 사건 수 ──
counts_by_tool = pd.DataFrame({
    '셔터 1 구간': df.Tool.value_counts(),
    '고장 기록(중복 제외)': events.Tool.value_counts(),
    '하루 묶음 사건': events.groupby('Tool').new.sum(),   # new=True인 기록 = 사건의 첫 기록
}).fillna(0).astype(int)
# 대표 6대는 그대로, 나머지 14대는 하나로 합친다
tool_group = counts_by_tool.index.where(counts_by_tool.index.isin(SEL), '나머지 14대')
counts_grouped = counts_by_tool.groupby(tool_group).sum().reindex(SEL + ['나머지 14대'])
measures = list(counts_by_tool.columns)

# ── 표: 개수와 전체 대비 비율(%) ─────────────────────────────────
share_table = counts_grouped.copy()
for measure in measures:
    share_table[measure + ' (%)'] = (counts_grouped[measure] / counts_grouped[measure].sum() * 100).round(1)
share_table.loc['대표 6대 합계'] = share_table.loc[SEL].sum()
share_table.loc['전체'] = share_table.loc[SEL + ['나머지 14대']].sum()
share_table[measures] = share_table[measures].astype(int)
display(share_table[[name for measure in measures for name in (measure, measure + ' (%)')]])

# ── 그림 0-1: 파이 3개를 가로로 배치 ──────────────────────────────
pie_colors = [TCOLOR[tool] for tool in SEL] + ['#dfe3e8']        # 나머지 14대는 연한 회색
pie_x_ranges = [(0, .30), (.35, .65), (.70, 1)]                  # 파이 3개의 가로 위치
fig = go.Figure()
for (x_left, x_right), measure in zip(pie_x_ranges, measures):
    counts = counts_grouped[measure]
    ratios = counts / counts.sum()
    fig.add_trace(go.Pie(labels=counts_grouped.index, values=counts, sort=False, direction='clockwise', hole=.35,
                         domain=dict(x=[x_left, x_right], y=[0, .78]),        # 위쪽 22%는 제목 자리로 비운다
                         marker=dict(colors=pie_colors, line=dict(color='white', width=1)),
                         # 3% 미만 조각은 글자가 겹치므로 숫자를 생략한다(마우스를 올리면 보임)
                         text=[f'{ratio:.1%}' if ratio >= .03 else '' for ratio in ratios], textinfo='text',
                         textposition='inside', name=measure,
                         hovertemplate='%{label}<br>%{value:,} (%{percent:.1%})<extra></extra>'))
    fig.add_annotation(x=(x_left + x_right) / 2, y=.9, xref='paper', yref='paper', showarrow=False, font=dict(size=14),
                       text=f'<b>{measure}</b><br>전체 {counts.sum():,} · 대표 6대 {ratios[SEL].sum():.1%}')
fig.update_layout(title='그림 0-1. 대표 6대가 전체에서 차지하는 비율 (회색 = 나머지 14대, 3% 미만 조각은 숫자 생략)',
                  legend=dict(orientation='h', x=.5, xanchor='center', y=-0.06))
show(fig, 600)

,셔터 1 구간,셔터 1 구간 (%),고장 기록(중복 제외),고장 기록(중복 제외) (%),하루 묶음 사건,하루 묶음 사건 (%)
Tool,,,,,,
03_M02,36084,6.9,356,29.4,140,24.6
09_M01,29980,5.7,150,12.4,99,17.4
08_M01,18529,3.5,117,9.7,56,9.8
04_M01,35000,6.7,9,0.7,4,0.7
07_M02,38400,7.3,6,0.5,4,0.7
10_M02,26168,5.0,2,0.2,1,0.2
나머지 14대,342107,65.0,571,47.2,265,46.6
대표 6대 합계,184161,35.1,640,52.9,304,53.4
전체,526268,100.1,1211,100.1,569,100.0


In [3]:
# 평소·z값·고장 전 정의는 eda_step 3-2와 같다

def nearest_fault_dist(segments, fault_events):
    """구간마다 가장 가까운 고장 기록(앞 또는 뒤)까지의 시간(초). 고장이 없는 장비는 무한대."""
    dist = np.full(len(segments), np.inf)
    seg_start, seg_end = segments.time_start.to_numpy(), segments.time_end.to_numpy()
    for tool, row_idx in segments.groupby('Tool').indices.items():
        fault_times = np.sort(fault_events.loc[fault_events.Tool == tool, 'time'].unique())
        if len(fault_times) == 0:
            continue
        # 구간 시작 직전 고장까지의 시간
        prev_pos = np.searchsorted(fault_times, seg_start[row_idx])
        dist_prev = np.where(prev_pos > 0, seg_start[row_idx] - fault_times[np.maximum(prev_pos - 1, 0)], np.inf)
        # 구간 끝 직후 고장까지의 시간
        next_pos = np.searchsorted(fault_times, seg_end[row_idx])
        dist_next = np.where(next_pos < len(fault_times),
                             fault_times[np.minimum(next_pos, len(fault_times) - 1)] - seg_end[row_idx], np.inf)
        dist[row_idx] = np.minimum(dist_prev, dist_next)
    return dist

# 평소 = 가장 가까운 고장 기록에서 24시간 넘게 떨어진 구간
is_normal = nearest_fault_dist(df, events) > DAY

# 조건 묶음 = 장비·레시피·스텝·셔터 (결측은 -1로 하나의 값처럼 취급)
df['_rs'] = df.recipe_step.fillna(-1)
df['_sh'] = df.shutter.fillna(-1)
ctx_id = df.groupby(['Tool', 'recipe', '_rs', '_sh'], sort=False).ngroup().to_numpy()

# 조건 묶음별 평소 평균·표준편차 (구간 하나를 1개로 셈, 가중치 없음)
normal_stats = df.loc[is_normal].groupby(ctx_id[is_normal])[FEAT].agg(['mean', 'std'])
has_enough_normal = pd.Series(ctx_id[is_normal]).value_counts().reindex(normal_stats.index).ge(30)   # 평소 구간 30개 미만이면 기준 없음

# z값 = (구간 평균 - 그 조건의 평소 평균) / 평소 표준편차
for sensor, mean_col in zip(KEY, FEAT):
    normal_mean = normal_stats[(mean_col, 'mean')].reindex(ctx_id).to_numpy()
    normal_std = (normal_stats[(mean_col, 'std')]
                  .where(has_enough_normal & (normal_stats[(mean_col, 'std')] > 1e-6))   # 표준편차 0이면 비움
                  .reindex(ctx_id).to_numpy())
    df['z_' + sensor] = ((df[mean_col].to_numpy() - normal_mean) / normal_std).astype('float32')
df['normal'] = is_normal
is_shutter1 = (df.shutter == 1).to_numpy()                               # 셔터 1 = 식각 중
is_prefault = {fault: (df[TTF[fault]] <= DAY).to_numpy() for fault in FAULTS}   # 고장 종류별 고장 전 24시간 구간

## 1. 20대 요약과 대표 장비 위치

20대 전체를 한 줄씩 요약한다. **압력 저하 전 냉각 압력**은 압력 저하 24시간 전 구간(셔터 1)의 냉각 압력 z값 평균이다(원본 행 수 가중). 음수면 고장 전에 평소보다 낮고, 양수면 높다. 고장 전 구간이 200개 미만이면 비운다.

In [4]:
by_tool = df.groupby('Tool')
# 장비 × 고장 종류별 기록 수 (중복 제외)
fault_counts = (events.pivot_table(index='Tool', columns='종류', values='time', aggfunc='size', fill_value=0)
                .reindex(columns=ORDER, fill_value=0))

# ── 압력 저하 24시간 전 냉각 압력 z: 장비별 가중 평균 ──────────────
# 대상: 셔터 1 · 압력 저하 24시간 전 · z값이 있는 구간
drop_fault = FAULTS[0]
z_col = 'z_FLOWCOOLPRESSURE'
use_rows = is_shutter1 & is_prefault[drop_fault] & df[z_col].notna().to_numpy()
# 가중치 = 구간의 원본 행 수(n_rows, 약 4초당 1행) = 구간이 이어진 시간 길이
#   z_x_weight = z × n_rows,  장비별 가중 평균 = Σ z_x_weight / Σ n_rows
#   → 몇 초짜리 짧은 구간이 몇 분짜리 긴 구간과 같은 무게를 갖지 않게 한다
prefault_z = (df.loc[use_rows].assign(z_x_weight=lambda seg: seg[z_col] * seg.n_rows).groupby('Tool')
              .agg(z_x_weight_sum=('z_x_weight', 'sum'), weight_sum=('n_rows', 'sum'), n_segments=('n_rows', 'size')))

# ── 20대 요약 표 ─────────────────────────────────────────────────
summary = pd.DataFrame({'구분': [tool[3:] for tool in by_tool.size().index], '셔터 1 구간': by_tool.size(),
                        '기간(일)': ((full_df.groupby('Tool').time_end.max() - tool_start_by) / DAY).round(0).astype(int)},
                       index=by_tool.size().index)
summary = summary.join(fault_counts).fillna(0)
summary['고장 기록'] = summary[ORDER].sum(axis=1).astype(int)
summary['하루 묶음 사건'] = events.groupby('Tool').new.sum().reindex(summary.index).fillna(0).astype(int)
summary['압력 저하 전 냉각 압력 z'] = ((prefault_z.z_x_weight_sum / prefault_z.weight_sum)
                                    .where(prefault_z.n_segments >= 200)          # 고장 전 구간 200개 미만이면 비움
                                    .reindex(summary.index).round(2))
summary['대표'] = summary.index.map(ROLE).fillna('')
summary = summary.sort_values('고장 기록', ascending=False)
summary[ORDER] = summary[ORDER].astype(int)
display(summary)

# 대표 6대가 전체에서 차지하는 비율
sel_summary = summary.loc[SEL]
print('대표 6대 / 전체: ' + ' · '.join(
    f'{measure} {sel_summary[measure].sum():,}/{summary[measure].sum():,} ({sel_summary[measure].sum() / summary[measure].sum():.1%})'
    for measure in ['고장 기록', '하루 묶음 사건', '셔터 1 구간']))

# ── 그림 1-1: 장비별 압력 저하 전 냉각 압력 z (작은 값부터 정렬, 대표 6대만 색칠) ──
bar_data = summary.dropna(subset=['압력 저하 전 냉각 압력 z']).sort_values('압력 저하 전 냉각 압력 z')
fig = go.Figure(go.Bar(x=bar_data.index, y=bar_data['압력 저하 전 냉각 압력 z'],
                       marker_color=[TCOLOR.get(tool, '#c9cdd2') for tool in bar_data.index],
                       text=[ROLE.get(tool, '') for tool in bar_data.index], textposition='outside',
                       hovertemplate='%{x}<br>z %{y:.2f}<extra></extra>'))
fig.add_hline(y=0, line=dict(color='#444', width=1))
fig.update_layout(title='그림 1-1. 압력 저하 24시간 전 냉각 압력 z (셔터 1) — 색칠한 막대가 대표 6대',
                  yaxis_title='평소 대비 z값', xaxis_title='')
show(fig, 420)

,구분,셔터 1 구간,기간(일),압력 저하,압력 과다,누설,고장 기록,하루 묶음 사건,압력 저하 전 냉각 압력 z,대표
Tool,,,,,,,,,,
03_M02,M02,36084,472,298,49,9,356,140,1.18,고장 최다 · 반대 패턴
09_M01,M01,29980,463,129,19,2,150,99,-0.18,고장 많은 M01
08_M01,M01,18529,414,68,32,17,117,56,-0.44,고장 많은 M01 · 다수 패턴
01_M02,M02,26856,423,53,40,16,109,28,0.49,
03_M01,M01,17301,444,40,51,0,91,36,-0.50,
09_M02,M02,13441,489,8,57,0,65,43,-0.12,
06_M02,M02,27128,432,28,5,26,59,13,0.19,
04_M02,M02,26385,446,21,1,15,37,18,-0.21,
07_M01,M01,41425,453,15,2,18,35,14,-0.04,


대표 6대 / 전체: 고장 기록 640/1,211 (52.8%) · 하루 묶음 사건 304/569 (53.4%) · 셔터 1 구간 184,161/526,268 (35.0%)


### 1-2. 20대 전체 보기

점을 전부 찍으면 52만 개라 아무것도 안 보인다. **요약해서 한 장에** 그린다. 모두 20대 전체, 샘플링 없음, 셔터 1 구간이다. 장비는 고장 기록이 많은 순으로 위에서부터 놓고, 이름 옆 괄호는 고장 기록 수다.

| 그림 | 보는 것 | 만드는 법 |
|---|---|---|
| 1-2, 1-3 장비 × 날짜 히트맵 | 냉각 압력·유량이 언제 평소보다 높거나 낮았는지, 고장 직전에 색이 바뀌는지 | 칸 = 장비 하루. 색 = 그날 z값의 원본 행 수 가중 평균(±3에서 자름, 빨강 = 평소보다 높음, 파랑 = 낮음, 빈칸 = 셔터 1 기록 없음). 검은 기호 = 고장 기록 |
| 1-4 고장 타임라인 | 고장이 언제, 어느 장비에 몰렸는지 | 장비별 고장 기록을 종류별 기호로 찍는다 |
| 1-5 장비별 평소 분포 | 장비마다 평소 수준과 흔들림 폭 | 평소 구간의 표준화된 원래 값으로 분위수(5·25·50·75·95%)를 계산해 상자만 그린다 |

가로축은 장비마다 **기록 시작 후 일수**다(장비마다 기록 시작 시각이 다르다).

In [5]:
# 장비 순서: 고장 기록 많은 순 (위 → 아래), 표시 이름에 고장 수
fault_total = events.Tool.value_counts().reindex(sorted(df.Tool.unique()), fill_value=0)
tool_order = fault_total.sort_values(ascending=False).index.tolist()
tool_label = {tool: f'{tool} ({fault_total[tool]})' for tool in tool_order}
day_of_row = ((df.time_start - df.Tool.map(tool_start_by)) // DAY).astype(int)   # 장비 기록 시작 후 몇째 날
FAULT_MARK = dict(zip(ORDER, ['#111111', '#6b21a8', '#0f5132']))                 # 히트맵 위 고장 기호 색 (진한 색)

def daily_matrix(sensor):
    # 장비 × 날짜 행렬: 하루 z값의 원본 행 수 가중 평균 = Σ(z × n_rows) / Σ(n_rows)
    z_values = df['z_' + sensor]
    has_z = z_values.notna()
    daily = (pd.DataFrame({'Tool': df.Tool[has_z], 'day': day_of_row[has_z],
                           'z_x_weight': z_values[has_z] * df.n_rows[has_z], 'weight': df.n_rows[has_z]})
             .groupby(['Tool', 'day'])[['z_x_weight', 'weight']].sum())
    daily_mean = (daily.z_x_weight / daily.weight).unstack('day')
    all_days = np.arange(day_of_row.max() + 1)
    return daily_mean.reindex(index=tool_order, columns=all_days)

for fig_no, sensor in [(2, 'FLOWCOOLPRESSURE'), (3, 'FLOWCOOLFLOWRATE')]:
    matrix = daily_matrix(sensor)
    fig = go.Figure(go.Heatmap(z=matrix.clip(-3, 3).to_numpy(), x=matrix.columns, y=[tool_label[tool] for tool in tool_order],
                               zmid=0, zmin=-3, zmax=3, colorscale='RdBu_r', colorbar=dict(title='z'),
                               hovertemplate='%{y}<br>%{x}일째<br>z %{z:.2f}<extra></extra>'))
    for ftype in ORDER:
        type_events = events[events.종류 == ftype]
        fig.add_trace(go.Scatter(x=(type_events.time - type_events.Tool.map(tool_start_by)) / DAY,
                                 y=type_events.Tool.map(tool_label), mode='markers', name=ftype,
                                 marker=dict(symbol=FSYMBOL[ftype], color=FAULT_MARK[ftype], size=6,
                                             line=dict(color='white', width=.5)),
                                 hovertemplate='%{y}<br>' + ftype + ' %{x:.1f}일째<extra></extra>'))
    fig.update_yaxes(categoryorder='array', categoryarray=[tool_label[tool] for tool in tool_order][::-1])
    fig.update_xaxes(title_text='장비 기록 시작 후 일수')
    fig.update_layout(title=f'그림 1-{fig_no}. 20대 {NAMES[sensor]} z값 — 장비 × 날짜 (하루 가중 평균, 셔터 1, 검은 기호 = 고장)',
                      legend=dict(orientation='h', y=-0.15))
    show(fig, 680)

# 그림 1-4: 고장 타임라인
fig = go.Figure()
for ftype in ORDER:
    type_events = events[events.종류 == ftype]
    fig.add_trace(go.Scatter(x=(type_events.time - type_events.Tool.map(tool_start_by)) / DAY,
                             y=type_events.Tool.map(tool_label), mode='markers', name=ftype,
                             marker=dict(symbol=FSYMBOL[ftype], color=FCOLOR[ftype], size=7, opacity=.7),
                             hovertemplate='%{y}<br>' + ftype + ' %{x:.1f}일째<extra></extra>'))
fig.update_yaxes(categoryorder='array', categoryarray=[tool_label[tool] for tool in tool_order][::-1])
fig.update_xaxes(title_text='장비 기록 시작 후 일수')
fig.update_layout(title='그림 1-4. 20대 고장 타임라인 (중복 제외 기록)', legend=dict(orientation='h', y=-0.15))
show(fig, 620)

# 고장이 몰린 정도: 장비별로 고장 기록이 가장 많은 30일 창에 든 비율
def busiest_30d_share(fault_times):
    fault_times = np.sort(fault_times)
    if len(fault_times) == 0:
        return np.nan
    window_end = np.searchsorted(fault_times, fault_times + 30 * DAY)
    return (window_end - np.arange(len(fault_times))).max() / len(fault_times) * 100
concentration = pd.DataFrame({
    '고장 기록': fault_total[tool_order],
    '기간(일)': ((full_df.groupby('Tool').time_end.max() - tool_start_by) / DAY).round(0).astype(int)[tool_order],
    '가장 붐빈 30일에 든 비율(%)': [round(busiest_30d_share(events.loc[events.Tool == tool, 'time'].to_numpy()), 1)
                               for tool in tool_order]})
display(concentration.rename_axis('장비 (고장 많은 순)'))

# 그림 1-5: 20대 평소 분포 (냉각 압력·유량, 표준화된 원래 값, 상자 25~75%, 수염 5~95%)
QUANTILES = [.05, .25, .5, .75, .95]
M_COLOR = {'M01': '#2a78d6', 'M02': '#eb6834'}
fig = make_subplots(rows=1, cols=2, subplot_titles=[NAMES['FLOWCOOLPRESSURE'], NAMES['FLOWCOOLFLOWRATE']])
spread_all = []
for col, sensor in enumerate(['FLOWCOOLPRESSURE', 'FLOWCOOLFLOWRATE'], 1):
    for tool in tool_order:
        values = df.loc[(df.Tool == tool) & df.normal, sensor + '_mean'].dropna()
        p05, p25, p50, p75, p95 = values.quantile(QUANTILES).to_numpy()
        spread_all.append({'장비': tool, '센서': NAMES[sensor], '중앙값': p50, '25~75% 폭': p75 - p25})
        fig.add_trace(go.Box(y=[tool_label[tool]], q1=[p25], median=[p50], q3=[p75], lowerfence=[p05], upperfence=[p95],
                             orientation='h', marker_color=M_COLOR[tool[3:]], name=tool[3:], legendgroup=tool[3:],
                             showlegend=(col == 1 and tool == next(t for t in tool_order if t[3:] == tool[3:])),
                             boxpoints=False), row=1, col=col)
    fig.update_yaxes(categoryorder='array', categoryarray=[tool_label[tool] for tool in tool_order][::-1], row=1, col=col)
fig.update_layout(title='그림 1-5. 20대 평소 분포 (셔터 1, 표준화된 원래 값) — 파랑 M01, 주황 M02', legend=dict(orientation='h', y=-0.1))
show(fig, 680)
spread_table = pd.DataFrame(spread_all).pivot(index='장비', columns='센서', values='25~75% 폭').reindex(tool_order).round(3)
spread_table.insert(0, '고장 기록', fault_total[tool_order])
display(spread_table.rename_axis(columns='평소 25~75% 폭'))
display(spread_table.assign(구분=[tool[3:] for tool in spread_table.index]).groupby('구분')
        [[NAMES['FLOWCOOLPRESSURE'], NAMES['FLOWCOOLFLOWRATE']]].median().round(3).rename_axis(columns='평소 25~75% 폭 중앙값'))

,고장 기록,기간(일),가장 붐빈 30일에 든 비율(%)
장비 (고장 많은 순),,,
03_M02,356,472,19.9
09_M01,150,463,16.0
08_M01,117,414,28.2
01_M02,109,423,24.8
03_M01,91,444,26.4
09_M02,65,489,20.0
06_M02,59,432,64.4
04_M02,37,446,35.1
07_M01,35,453,37.1


평소 25~75% 폭,고장 기록,냉각 압력,냉각 유량
장비,,,
03_M02,356,0.026,0.005
09_M01,150,0.020,0.001
08_M01,117,0.174,0.001
01_M02,109,0.376,0.855
03_M01,91,0.118,0.001
09_M02,65,0.076,0.001
06_M02,59,0.409,0.915
04_M02,37,0.577,1.001
07_M01,35,0.028,0.001


평소 25~75% 폭 중앙값,냉각 압력,냉각 유량
구분,,
M01,0.024,0.001
M02,0.056,0.003


## 2. 대표 6대 개요

셔터 1 비율은 모든 셔터 기록 시간 중 셔터 1 시간, 평소 비율은 셔터 1 시간 중 평소 시간이다(원본 행 수 기준). 레시피 수는 장비에서 한 번이라도 나온 레시피 개수다.

In [6]:
sel_df = df[df.Tool.isin(SEL)]           # 이후 모든 장에서 쓰는 대표 6대 데이터 (전체 구간)

# ── 6대 개요 표 ──────────────────────────────────────────────────
overview_rows = []
for tool in SEL:
    tool_df = sel_df[sel_df.Tool == tool]
    full_tool = full_df[full_df.Tool == tool]          # 모든 셔터 구간 (셔터 1 비율·기간용)
    weight = tool_df.n_rows          # 비율 계산의 가중치 = 구간의 원본 행 수(시간 길이) → "시간 기준" 비율
    tool_events = events[events.Tool == tool]
    overview_rows.append({
        '장비': tool, '역할': ROLE[tool],
        '기간(일)': round((full_tool.time_end.max() - full_tool.time_start.min()) / DAY),
        '셔터 1 구간': len(tool_df),
        '셔터 1 비율(%)': round(full_tool.n_rows[full_tool.shutter == 1].sum() / full_tool.n_rows.sum() * 100, 1),   # 셔터 1 시간 / 전체 시간
        '평소 비율(%)': round(weight[tool_df.normal].sum() / weight.sum() * 100, 1),          # 평소 시간 / 셔터 1 시간
        '레시피 수': tool_df.recipe.nunique(),
        **{ftype: int((tool_events.종류 == ftype).sum()) for ftype in ORDER},
        '하루 묶음 사건': int(tool_events.new.sum())})
overview = pd.DataFrame(overview_rows).set_index('장비')
display(overview)

# ── 레시피 겹침: 자카드 = 공통 레시피 수 / 두 장비 레시피 합집합 수 ──
recipes_by_tool = {tool: set(sel_df.loc[sel_df.Tool == tool, 'recipe'].astype(str)) for tool in SEL}
recipe_jaccard = pd.DataFrame([[len(recipes_by_tool[a] & recipes_by_tool[b]) / len(recipes_by_tool[a] | recipes_by_tool[b])
                                for b in SEL] for a in SEL], index=SEL, columns=SEL).round(2)
display(Markdown('**레시피 겹침 (자카드: 두 장비 레시피 중 공통 비율)**'))
display(recipe_jaccard)

,역할,기간(일),셔터 1 구간,셔터 1 비율(%),평소 비율(%),레시피 수,압력 저하,압력 과다,누설,하루 묶음 사건
장비,,,,,,,,,,
03_M02,고장 최다 · 반대 패턴,472,36084,55.2,35.8,57,298,49,9,140
09_M01,고장 많은 M01,463,29980,55.5,58.9,58,129,19,2,99
08_M01,고장 많은 M01 · 다수 패턴,414,18529,64.0,76.3,25,68,32,17,56
04_M01,고장 적은 M01,467,35000,56.5,98.9,72,7,1,1,4
07_M02,고장 적은 M02,430,38400,53.6,98.3,54,3,3,0,4
10_M02,고장 최소 M02,438,26168,53.7,99.5,96,2,0,0,1


**레시피 겹침 (자카드: 두 장비 레시피 중 공통 비율)**

,03_M02,09_M01,08_M01,04_M01,07_M02,10_M02
03_M02,1.00,0.00,0.00,0.00,0.16,0.19
09_M01,0.00,1.00,0.01,0.26,0.00,0.00
08_M01,0.00,0.01,1.00,0.01,0.00,0.00
04_M01,0.00,0.26,0.01,1.00,0.00,0.00
07_M02,0.16,0.00,0.00,0.00,1.00,0.29
10_M02,0.19,0.00,0.00,0.00,0.29,1.00


## 3. 전체 기간 추세

장비별 셔터 1 구간의 z값을 **6시간 단위로 묶어 원본 행 수로 가중 평균**했다. 0은 그 장비·레시피·스텝의 평소 수준이다. 아래쪽 기호는 고장 기록이다(▼ 압력 저하, ▲ 압력 과다, × 누설). 세로축은 ±6에서 자른다.

**가중치**: 스텝 구간마다 그 구간의 **원본 행 수**(`n_rows`, 약 4초당 1행)를 가중치로 준다. 원본 행 수는 구간이 이어진 시간 길이와 같으므로, 6시간 칸의 값은 Σ(z × 원본 행 수) / Σ(원본 행 수)이다. 예를 들어 z = +2.0인 4초짜리 구간과 z = 0.0인 6분짜리 구간이 있으면 단순 평균은 1.0이지만 가중 평균은 0.02다. 몇 초짜리 짧은 구간이 긴 구간과 같은 무게를 갖지 않게 하려는 것이다. 1장(압력 저하 전 냉각 압력 z)과 6장(고장 전 48시간 궤적)도 같은 방식이다.

In [7]:
BIN_SEC = 6 * HOUR                        # 긴 기간 그림의 묶음 단위: 6시간

def six_hour_mean(tool, sensor):
    """장비·센서의 z값을 6시간 칸별 가중 평균한다. 반환: (기록 시작 후 일수를 index로 한 Series, 장비 시작 시각)"""
    seg = sel_df[(sel_df.Tool == tool) & (sel_df.shutter == 1)].dropna(subset=['z_' + sensor])   # 셔터 1 · z값 있는 구간만
    tool_start = tool_start_by[tool]                                                             # 장비 기록 시작 시각
    bin_idx = (seg.time_start - tool_start) // BIN_SEC          # 구간이 속한 6시간 칸 번호 (묶는 기준, 가중치 아님)
    weight = seg.n_rows                                         # 가중치 = 구간의 원본 행 수(구간 길이)
    # 칸별 가중 평균 = Σ(z × weight) / Σ(weight)
    #   예: z=+2.0(1행, 4초)과 z=0.0(99행, 약 6분) → 단순 평균 1.0, 가중 평균 0.02
    bin_mean = (seg['z_' + sensor] * weight).groupby(bin_idx).sum() / weight.groupby(bin_idx).sum()
    return pd.Series(bin_mean.values, index=bin_mean.index * BIN_SEC / DAY), tool_start   # 칸 번호 -> 일수

# ── 그림 3-1(냉각 압력), 3-2(냉각 유량): 장비 4대를 위아래로 ──────
for sensor in ['FLOWCOOLPRESSURE', 'FLOWCOOLFLOWRATE']:
    fig = make_subplots(rows=len(SEL), cols=1, shared_xaxes=False, vertical_spacing=.04,
                        subplot_titles=[f'{tool} ({ROLE[tool]})' for tool in SEL])
    for row, tool in enumerate(SEL, 1):
        z_6h, tool_start = six_hour_mean(tool, sensor)
        # 선: 6시간 가중 평균 z (±6에서 자름). Scattergl = 점이 많아도 가벼운 WebGL 그림
        fig.add_trace(go.Scattergl(x=z_6h.index, y=z_6h.values.clip(-6, 6), mode='lines',
                                   line=dict(color=TCOLOR[tool], width=1), showlegend=False,
                                   hovertemplate='%{x:.1f}일<br>z %{y:.2f}<extra></extra>'), row=row, col=1)
        # 아래쪽 기호: 고장 기록 위치 (종류별 모양·색)
        tool_events = events[events.Tool == tool]
        for ftype in ORDER:
            type_events = tool_events[tool_events.종류 == ftype]
            fig.add_trace(go.Scattergl(x=(type_events.time - tool_start) / DAY, y=np.full(len(type_events), -5.4),
                                       mode='markers', marker=dict(symbol=FSYMBOL[ftype], color=FCOLOR[ftype], size=7),
                                       name=ftype, legendgroup=ftype, showlegend=row == 1,
                                       hovertemplate=ftype + ' %{x:.1f}일<extra></extra>'), row=row, col=1)
        fig.add_hline(y=0, line=dict(color='#999', dash='dot', width=1), row=row, col=1)   # 0 = 평소 수준
        fig.update_yaxes(range=[-6, 6], row=row, col=1)
    fig.update_xaxes(title_text='장비 기록 시작 후 일수', row=len(SEL), col=1)
    fig.update_layout(title=f'그림 3-{1 if sensor == "FLOWCOOLPRESSURE" else 2}. {NAMES[sensor]} z값 6시간 평균 (셔터 1)',
                      legend=dict(orientation='h', y=-0.08))
    show(fig, 1250)

## 4. 평소 상태 분포

평소 구간(고장에서 24시간 넘게 떨어짐) 중 셔터 1만 쓴다. z값은 장비마다 평소를 0으로 맞춘 값이라 비교 의미가 없으므로, **표준화된 원래 값**을 비교한다. 상자는 25~75%, 수염은 5~95%, 가운데 선은 중앙값이다(구간 수 기준). M01과 M02는 레시피가 달라서 수준 차이에 레시피 차이가 섞여 있다.

In [8]:
QUANTILES = [.05, .25, .5, .75, .95]          # 상자 그림에 쓸 분위수 (수염 5%, 상자 25~75%, 중앙값 50%)
fig = make_subplots(rows=2, cols=3, subplot_titles=[NAMES[sensor] for sensor in KEY], vertical_spacing=.14)
spread_rows = []
for pos, sensor in enumerate(KEY):
    grid_row, grid_col = pos // 3 + 1, pos % 3 + 1
    for tool in SEL:
        # 평소 · 셔터 1 구간의 표준화된 원래 값 (구간 하나를 1개로 셈, 가중치 없음)
        values = sel_df.loc[(sel_df.Tool == tool) & sel_df.normal & (sel_df.shutter == 1), sensor + '_mean'].dropna()
        p05, p25, p50, p75, p95 = values.quantile(QUANTILES).to_numpy()
        spread_rows.append({'센서': NAMES[sensor], '장비': tool, '중앙값': p50, '25~75% 폭': p75 - p25})
        # 분위수를 미리 계산해 상자만 그린다 (수십만 점을 그림에 넣지 않음)
        fig.add_trace(go.Box(x=[tool], q1=[p25], median=[p50], q3=[p75], lowerfence=[p05], upperfence=[p95],
                             marker_color=TCOLOR[tool], name=tool, showlegend=False, boxpoints=False),
                      row=grid_row, col=grid_col)
fig.update_layout(title='그림 4-1. 평소 상태 분포 (셔터 1, 표준화된 원래 값) — 상자 25~75%, 수염 5~95%')
show(fig, 640)
# 평소 흔들림 폭 표 (25~75% 폭, 작을수록 안정)
display(pd.DataFrame(spread_rows).pivot(index='센서', columns='장비', values='25~75% 폭')[SEL].round(3)
        .rename_axis(columns='평소 25~75% 폭(작을수록 안정)'))

평소 25~75% 폭(작을수록 안정),03_M02,09_M01,08_M01,04_M01,07_M02,10_M02
센서,,,,,,
냉각 압력,0.026,0.020,0.174,0.122,0.008,0.036
냉각 유량,0.005,0.001,0.001,0.000,0.001,0.000
빔 전류,1.135,0.004,0.001,0.004,0.006,0.007
아르곤 유량(소스),0.552,0.009,0.002,0.003,0.002,0.000
억제 전류,0.547,0.409,0.110,0.190,0.150,0.413
챔버 압력,0.295,0.199,0.054,0.157,0.110,0.254


## 5. 고장 간격과 반복

중복을 뺀 고장 기록마다 **같은 장비의 바로 앞 고장 기록(종류 무관)까지 걸린 시간**을 잰다. 1시간 안에 다시 찍힌 기록은 같은 고장이 반복 기록됐을 가능성이 크다.

In [9]:
# 고장 기록마다 같은 장비의 바로 앞 기록(종류 무관)까지 걸린 시간
intervals = events[events.Tool.isin(SEL)].copy()
intervals['간격(시간)'] = intervals.groupby('Tool').time.diff() / HOUR      # 장비 첫 기록은 앞 기록이 없어 NaN

def share_below(gap_hours, limit_h):
    """간격이 limit_h시간 미만인 기록 비율(%). 분모 = 간격이 있는 기록 수."""
    return round((gap_hours < limit_h).sum() / max(gap_hours.notna().sum(), 1) * 100, 1)

# ── 간격 요약 표 ─────────────────────────────────────────────────
interval_table = intervals.groupby('Tool').agg(
    고장기록=('time', 'size'), 사건=('new', 'sum'), 간격중앙값_시간=('간격(시간)', 'median'),
    **{'1시간 안 재기록(%)': ('간격(시간)', lambda gap_hours: share_below(gap_hours, 1))},
    **{'1일 안 재기록(%)': ('간격(시간)', lambda gap_hours: share_below(gap_hours, 24))}).reindex(SEL)
display(interval_table.round(1))

# ── 그림 5-1: 간격 분포 (가로축 로그 눈금, 1분 미만은 1분으로 맞춤) ──
plot_data = intervals.dropna(subset=['간격(시간)'])
fig = go.Figure()
for ftype in ORDER:
    type_data = plot_data[plot_data.종류 == ftype]
    fig.add_trace(go.Scatter(x=np.log10(type_data['간격(시간)'].clip(lower=1 / 60)), y=type_data.Tool, mode='markers',
                             name=ftype, marker=dict(color=FCOLOR[ftype], symbol=FSYMBOL[ftype], size=8, opacity=.6),
                             customdata=type_data['간격(시간)'],
                             hovertemplate='%{y}<br>%{customdata:.1f}시간<extra>' + ftype + '</extra>'))
fig.update_traces(marker_line_width=0)
fig.update_xaxes(tickvals=np.log10([1 / 60, 1, 24, 24 * 7, 24 * 30, 24 * 180]),
                 ticktext=['1분', '1시간', '1일', '1주', '1달', '6달'], title_text='앞 고장 기록부터 걸린 시간 (로그 눈금)')
fig.update_yaxes(categoryorder='array', categoryarray=SEL[::-1])
fig.update_layout(title='그림 5-1. 고장 기록 간격 (색 = 뒤 기록의 고장 종류)', legend=dict(orientation='h', y=-0.2))
show(fig, 420)

,고장기록,사건,간격중앙값_시간,1시간 안 재기록(%),1일 안 재기록(%)
Tool,,,,,
03_M02,356,140,17.3,6.8,60.8
09_M01,150,99,39.1,2.7,34.2
08_M01,117,56,21.6,9.5,52.6
04_M01,9,4,12.4,12.5,62.5
07_M02,6,4,56.8,40.0,40.0
10_M02,2,1,23.7,0.0,100.0


## 6. 고장 전 48시간 궤적 — 장비별 비교

반복 기록이 앞 고장의 영향을 받지 않도록 **하루 묶음 사건의 첫 기록만** 쓴다. 첫 기록마다 고장 전 48시간을 1시간 단위로 나눠 셔터 1 구간의 z값을 원본 행 수로 가중 평균하고, 장비별로 **사건 간 중앙값**을 그린다. 사건이 1~2개인 장비는 선이 크게 흔들리므로 표의 사건 수를 함께 본다.

In [10]:
TRAJ_SENSORS = ['FLOWCOOLPRESSURE', 'FLOWCOOLFLOWRATE']    # 궤적을 볼 센서
WINDOW_H = 48                                              # 고장 전 몇 시간을 볼지
# 하루 묶음 사건의 첫 기록만 쓴다 (반복 기록은 앞 고장의 영향을 받으므로 제외)
first_records = events[events.new & events.Tool.isin(SEL)]

# ── 사건마다 고장 전 48시간을 1시간 칸으로 나눠 가중 평균 ──────────
traj_parts = []
for tool in SEL:
    seg = sel_df[(sel_df.Tool == tool) & (sel_df.shutter == 1)]   # 셔터 1 구간만
    seg_start = seg.time_start.to_numpy()
    weight = seg.n_rows.to_numpy()                                # 가중치 = 구간의 원본 행 수(구간 길이)
    for fault in first_records[first_records.Tool == tool].itertuples():
        # 고장 48시간 전 ~ 고장 직전 구간의 위치 범위
        win_lo, win_hi = np.searchsorted(seg_start, [fault.time - WINDOW_H * HOUR, fault.time])
        if win_hi <= win_lo:
            continue
        hours_before = (fault.time - seg_start[win_lo:win_hi] - 1) // HOUR   # 고장까지 몇 시간 전 칸인지 (0 = 직전 1시간)
        win_weight = weight[win_lo:win_hi]
        for sensor in TRAJ_SENSORS:
            z_vals = seg['z_' + sensor].to_numpy()[win_lo:win_hi]
            has_z = np.isfinite(z_vals)
            if has_z.sum() == 0:
                continue
            # 칸별 가중 평균 = Σ(z × weight) / Σ(weight)
            hourly_mean = (pd.Series(z_vals[has_z] * win_weight[has_z]).groupby(hours_before[has_z]).sum()
                           / pd.Series(win_weight[has_z]).groupby(hours_before[has_z]).sum())
            traj_parts.append(pd.DataFrame({'Tool': tool, '종류': fault.종류, 'event': fault.time, 'sensor': sensor,
                                            'h': hourly_mean.index, 'z': hourly_mean.values}))
traj = pd.concat(traj_parts, ignore_index=True)
# 사건 사이에서는 가중치 없이 중앙값 (사건 하나를 1개로 셈)
traj_median = traj.groupby(['Tool', '종류', 'sensor', 'h']).z.median().reset_index()
n_events = (traj.groupby(['Tool', '종류']).event.nunique().unstack()
            .reindex(index=SEL, columns=ORDER).fillna(0).astype(int))
display(n_events.rename_axis(columns='궤적에 쓴 사건 수(첫 기록)'))

# ── 그림 6-1: 행 = 센서, 열 = 고장 종류, 선 = 장비 ─────────────────
fig = make_subplots(rows=2, cols=3, shared_xaxes=True, vertical_spacing=.1,
                    subplot_titles=[f'{NAMES[sensor]} · {ftype}' for sensor in TRAJ_SENSORS for ftype in ORDER])
legend_shown = set()                              # 범례에 장비 이름을 한 번만 넣기 위한 기록
for row, sensor in enumerate(TRAJ_SENSORS, 1):
    for col, ftype in enumerate(ORDER, 1):
        for tool in SEL:
            line_data = traj_median[(traj_median.Tool == tool) & (traj_median.종류 == ftype)
                                    & (traj_median.sensor == sensor)].sort_values('h')
            if line_data.empty:
                continue
            n_ev = n_events.loc[tool, ftype]
            # 사건 3개 미만이면 가는 점선 (흔들림이 커서 믿기 어려움)
            fig.add_trace(go.Scatter(x=-line_data.h - .5, y=line_data.z, mode='lines', name=tool, legendgroup=tool,
                                     showlegend=tool not in legend_shown,
                                     line=dict(color=TCOLOR[tool], width=2 if n_ev >= 3 else 1,
                                               dash='solid' if n_ev >= 3 else 'dot'),
                                     hovertemplate=f'{tool} (사건 {n_ev}개)<br>' + '%{x:.1f}시간 전<br>z %{y:.2f}<extra></extra>'),
                          row=row, col=col)
            legend_shown.add(tool)
        fig.add_hline(y=0, line=dict(color='#999', dash='dot', width=1), row=row, col=col)
fig.update_xaxes(title_text='고장까지 남은 시간(시간)', row=2)
fig.update_layout(title='그림 6-1. 고장 전 48시간 z값 — 장비별 사건 간 중앙값 (점선 = 사건 3개 미만)',
                  legend=dict(orientation='h', y=-0.15))
show(fig, 680)

# ── 직전 변화량 표: (직전 3시간 중앙값) − (24~48시간 전 중앙값) ─────
last_3h = traj_median[traj_median.h < 3].groupby(['Tool', '종류', 'sensor']).z.median()
early_24_48h = traj_median[traj_median.h >= 24].groupby(['Tool', '종류', 'sensor']).z.median()
late_change = (last_3h - early_24_48h).rename('직전 3시간 − 24~48시간 전').reset_index()
late_change['센서'] = late_change.sensor.map(NAMES)
display(late_change.pivot_table(index=['종류', 'Tool'], columns='센서', values='직전 3시간 − 24~48시간 전').round(2))

궤적에 쓴 사건 수(첫 기록),압력 저하,압력 과다,누설
Tool,,,
03_M02,117,17,2
09_M01,81,17,1
08_M01,43,9,3
04_M01,3,1,0
07_M02,3,1,0
10_M02,1,0,0


센서            냉각 압력  냉각 유량
종류    Tool                
누설    03_M02   0.07  -0.29
      08_M01   0.77   0.37
      09_M01  -1.41  -0.05
압력 과다 03_M02   1.14   1.07
      07_M02  -0.21   0.15
      08_M01  -0.06  -0.70
      09_M01   0.02   0.80
압력 저하 03_M02   0.02   0.23
      04_M01   0.23  -0.07
      07_M02   0.20   0.93
      08_M01  -0.01   0.42
      09_M01  -0.22  -0.02
      10_M02  -1.41   0.33

## 7. 사례 확대 — 원래 해상도

장비마다 **고장 전 48시간 기록이 가장 많은 사건(첫 기록)** 하나를 골라, 고장 48시간 전부터 6시간 뒤까지 셔터 1 구간을 **줄이지 않고** 그린다(점 하나 = 스텝 구간 하나). 세로 점선이 고장 시각이다.

In [11]:
# ── 장비마다 사례 1건 고르기: 고장 전 48시간에 셔터 1 구간이 가장 많은 첫 기록 ──
cases = {}
for tool in SEL:
    seg_start = sel_df.loc[(sel_df.Tool == tool) & (sel_df.shutter == 1), 'time_start'].to_numpy()
    candidates = first_records[first_records.Tool == tool]
    n_seg_48h = [np.diff(np.searchsorted(seg_start, [fault_time - 48 * HOUR, fault_time]))[0]
                 for fault_time in candidates.time]                      # 사건별 48시간 안 구간 수
    best = int(np.argmax(n_seg_48h))
    cases[tool] = (candidates.iloc[best], int(n_seg_48h[best]))

# ── 그림 7-1: 고장 48시간 전 ~ 6시간 뒤, 줄이지 않은 원래 해상도 (점 하나 = 스텝 구간 하나, 평균 없음) ──
fig = make_subplots(rows=len(SEL), cols=1, vertical_spacing=.05, shared_xaxes=True,
                    subplot_titles=[f'{tool} ({ROLE[tool]}) · {cases[tool][0].종류}' for tool in SEL])
case_rows = []
for row, tool in enumerate(SEL, 1):
    seg = sel_df[(sel_df.Tool == tool) & (sel_df.shutter == 1)]
    fault, n_before = cases[tool]
    window = seg[(seg.time_start >= fault.time - 48 * HOUR) & (seg.time_start <= fault.time + 6 * HOUR)]
    hours_from_fault = (window.time_start - fault.time) / HOUR           # 고장 시각을 0으로 한 시간(시간 단위)
    case_rows.append({'장비': tool, '고장 종류': fault.종류, '48시간 전 구간 수': n_before, '그린 점': len(window)})
    for sensor, color in zip(TRAJ_SENSORS, ['#2a78d6', '#eb6834']):
        fig.add_trace(go.Scattergl(x=hours_from_fault, y=window['z_' + sensor].clip(-10, 10), mode='markers',
                                   name=NAMES[sensor], legendgroup=sensor, showlegend=row == 1,
                                   marker=dict(color=color, size=4, opacity=.6),
                                   hovertemplate='%{x:.2f}시간<br>z %{y:.2f}<extra>' + NAMES[sensor] + '</extra>'),
                      row=row, col=1)
    fig.add_vline(x=0, line=dict(color='#444', dash='dash', width=1), row=row, col=1)   # 고장 시각
    fig.add_hline(y=0, line=dict(color='#999', dash='dot', width=1), row=row, col=1)    # 평소 수준
fig.update_xaxes(title_text='고장 시각 기준 시간(시간)', row=len(SEL), col=1)
fig.update_yaxes(title_text='z')
fig.update_layout(title='그림 7-1. 고장 전후 사례 (셔터 1, 원래 해상도, z는 ±10에서 자름)', legend=dict(orientation='h', y=-0.08))
show(fig, 1250)
display(pd.DataFrame(case_rows).set_index('장비'))

,고장 종류,48시간 전 구간 수,그린 점
장비,,,
03_M02,압력 저하,267,283
09_M01,압력 저하,235,237
08_M01,압력 저하,179,209
04_M01,압력 저하,150,150
07_M02,압력 저하,232,248
10_M02,압력 저하,120,121


## 8. 해석

**결론: 냉각 압력은 고장과 상관없이 모든 장비에서 몇 달 단위로 수준이 오르내린다. 고장 전 압력의 "방향"은 고장 직전 변화라기보다 그 시기의 수준이고, 장비마다 공정·흔들림 폭도 달라 하나의 기준으로 처음 보는 장비를 맞히기 어렵다. 고장 많은 장비는 평소보다 벗어난 날이 더 많다.**

### 20대 전체에서 보인 것 (1-2절)

| 관찰 | 근거 |
|---|---|
| **냉각 압력 수준 이동은 고장 수와 무관하게 모든 장비에 있다** | 30일 평균 z의 최소~최대가 거의 모든 장비에서 −1 ~ +1 이상 벌어진다. 고장 2건뿐인 10_M02도 −0.88 ~ +1.75, 19건인 01_M01은 최대 +2.62 (그림 1-2) |
| **고장 많은 장비는 평소에서 벗어난 날이 많다** | 하루 평균 냉각 압력 \|z\| > 1인 날: 고장 상위 3대 32~51%, 고장 9건 이하 4대 4~28%. 냉각 유량도 상위 3대 13~17%, 하위 4대 1~9%. 예외로 05_M02(27건)는 압력 60% |
| **고장이 한 시기에 몰린 장비가 있다** | 가장 붐빈 30일에 든 고장 비율: 10_M01 80%, 06_M02 64%. 고장 상위 3대는 16~28%로 기간 전체에 퍼져 있다 (그림 1-4) |
| **평소 흔들림 폭은 M01/M02로 갈리지 않는다** | 냉각 압력 25~75% 폭 중앙값 M01 0.024, M02 0.056으로 비슷하다. 대신 M02 세 대(01_M02, 06_M02, 04_M02)만 냉각 압력 0.38~0.58, 냉각 유량 0.86~1.00으로 다른 장비보다 수십~수백 배 넓다 (그림 1-5) |

### 질문별 답 (대표 6대 포함)

| 질문 | 답 | 정한 것 |
|---|---|---|
| Q1. 고장 전 압력 방향이 같은가? | **아니다.** 값이 있는 16대 중 11대는 낮고 5대는 높다. M01은 10대 중 9대가 낮고, M02는 6대 중 4대가 높다 (1장) | 공통 모델 하나보다 **M01/M02별 또는 장비별** 기준이 맞다 |
| Q2. 직전 변화인가, 수준 차이인가? | **압력 저하는 수준 차이다.** 직전 3시간 − 24~48시간 전 냉각 압력 변화가 사건이 많은 3대에서 −0.22 ~ +0.02로 거의 0이다. 반면 위 1-2절처럼 수준은 몇 달 단위로 크게 움직인다 | 입력을 현재 수준이 아니라 **최근 대비 변화**(예: 최근 6시간 − 최근 7일)로 시험한다 |
| Q3. 고장 많은 장비는 평소부터 다른가? | **평소 흔들림 폭은 아니고, 벗어난 날의 수는 그렇다.** 대표 6대의 평소 냉각 압력 폭은 08_M01 0.17, 04_M01 0.12가 넓고 나머지는 0.01~0.04로 고장 수와 관계없다. 03_M02는 빔 전류·아르곤 유량 폭이 다른 장비의 100배 이상이다 | 경보 기준은 **장비별로** 정한다. "평소에서 벗어난 날의 비율"을 장비 상태 지표로 검토한다 |
| Q4. 레시피가 겹치는가? | **거의 안 겹친다.** 대표 6대 사이 자카드 0.00~0.29. 같은 종류 안에서 겹치는 쌍(09_M01–04_M01 0.26, 07_M02–10_M02 0.29)도 있지만 대부분 0.01 이하다 | 처음 보는 장비로 일반화하기 어렵다. 레시피는 입력이 아니라 **z값 기준**으로만 쓴다 |
| Q5. 반복 기록은 장비마다 다른가? | **고장 많은 장비끼리도 다르다.** 1일 안 재기록이 03_M02 60.8%, 08_M01 52.6%, 09_M01 34.2%다(고장 적은 장비는 기록이 2~9건이라 비교 어려움) | 하루 묶음 사건과 직후 경보 규칙은 **모든 장비에 같이** 쓰되, 반복 정도가 장비마다 다르다는 점을 평가에서 본다 |

**그 밖의 관찰**
- 03_M02는 셔터 1 시간 중 평소가 35.8%뿐이다(고장 적은 3대는 98~100%). 고장 전 구간이 특정 시기에 몰려 Q1의 "압력 방향"이 그 시기의 수준을 반영했을 수 있다.
- 압력 과다 직전에는 냉각 유량이 움직이지만 방향이 장비마다 다르다: 03_M02 +1.07(사건 17개), 09_M01 +0.80(17개), 08_M01 −0.70(9개).
- 누설 직전 냉각 압력: 08_M01 +0.77(사건 3개), 09_M01 −1.41(1개). 사건 수가 적어 확정할 수 없다.

**한계**
- 고장 적은 3대(04_M01 사건 4개, 07_M02 4개, 10_M02 1개)는 고장 전 궤적이 사실상 사례 몇 개다.
- 평소 기준을 전체 기간으로 만들었으므로, 수준 이동이 z값에 그대로 나타난다. 이는 의도한 것이지만, 모델 입력으로 쓸 때는 과거 데이터만으로 기준을 다시 만들어야 한다.
- 히트맵 가로축은 장비별 기록 시작 후 일수라, 같은 열이 같은 달력 날짜는 아니다.
- 고장 기록 수는 중복을 뺀 값이라 원본 기록 수(1,236)와 조금 다르다(03_M02 361 → 356).